In [13]:
import pandas as pd
import os

def calculate_sdi_v7():
    # ==========================================
    # 1. 配置文件路径
    # ==========================================
    base_path = r"C:\Users\qyw\Desktop\智能体\各智能体\基金经理分析智能体\数据\处理后\SDI"

    file_nav = os.path.join(base_path, "Fund_NAV1.csv")
    file_rf = os.path.join(base_path, "FUND_RISKFREE.xlsx")
    file_info = os.path.join(base_path, "基金主要信息.xlsx")
    file_output = os.path.join(base_path, "Fund_SDI_Results.xlsx")

    print("正在加载数据...")
    # ==========================================
    # 2. 读取并清洗数据
    # ==========================================
    # A. 读取交易数据 (NAV)
    try:
        df_nav = pd.read_csv(file_nav, encoding='utf-8-sig')
    except:
        df_nav = pd.read_csv(file_nav, encoding='gbk')

    # 自动匹配列名
    new_cols_nav = {}
    for col in df_nav.columns:
        c = str(col).strip()
        if "Week" in c: new_cols_nav[col] = "TradingWeek"
        elif "MasterFundCode" in c: new_cols_nav[col] = "MasterFundCode"
        elif "ReturnNAV" in c: new_cols_nav[col] = "ReturnNAV"
    df_nav.rename(columns=new_cols_nav, inplace=True)

    # 补齐代码并初步去重交易数据（保险起见）
    df_nav['MasterFundCode'] = df_nav['MasterFundCode'].astype(str).str.split('.').str[0].str.zfill(6)
    df_nav = df_nav.drop_duplicates(subset=['MasterFundCode', 'TradingWeek'])

    # B. 读取并清洗【基金主要信息】(针对你发现的重复问题进行处理)
    df_info = pd.read_excel(file_info)
    df_info['MasterFundCode'] = df_info['MasterFundCode'].astype(str).str.split('.').str[0].str.zfill(6)

    # 【核心修改】：针对基金主要信息进行去重
    # 如果同一个基金代码有多个分类，默认保留第一条；你也可以根据需要调整去重逻辑
    info_before = len(df_info)
    df_info = df_info.drop_duplicates(subset=['MasterFundCode'], keep='first')
    info_after = len(df_info)
    if info_before > info_after:
        print(f"⚠️ 检测到【基金主要信息】中存在 {info_before - info_after} 条重复记录，已自动剔除。")

    # C. 读取无风险利率
    df_rf = pd.read_excel(file_rf)

    # ==========================================
    # 3. 日期处理与数据合并
    # ==========================================
    print("正在处理日期并匹配利率...")
    df_nav['Date_Temp'] = pd.to_datetime(df_nav['TradingWeek'].astype(str) + '0', format='%Y%W%w', errors='coerce')
    df_nav = df_nav.dropna(subset=['Date_Temp', 'MasterFundCode', 'ReturnNAV'])

    df_rf['TradingDate'] = pd.to_datetime(df_rf['TradingDate'])

    # 排序
    df_nav = df_nav.sort_values('Date_Temp')
    df_rf = df_rf.sort_values('TradingDate')

    # 匹配利率
    df_merged = pd.merge_asof(
        df_nav, df_rf,
        left_on='Date_Temp', right_on='TradingDate',
        direction='backward'
    )

    # 计算周超额收益
    df_merged['Excess_Return'] = df_merged['ReturnNAV'] - df_merged['InterestRateWeekly']

    # 合并基金分类信息
    # 此时 df_info 已经唯一，不会再导致 obs_count 翻倍
    df_merged = pd.merge(
        df_merged,
        df_info[['MasterFundCode', 'Category', 'InvestmentStyle']],
        on='MasterFundCode',
        how='left'
    )
    df_merged = df_merged.dropna(subset=['Category', 'InvestmentStyle', 'Excess_Return'])

    # ==========================================
    # 4. 计算 SDI 指数
    # ==========================================
    print("正在计算同类基准收益及 SDI...")
    peer_cols = ['TradingWeek', 'Category', 'InvestmentStyle']
    peer_avg = df_merged.groupby(peer_cols)['Excess_Return'].mean().reset_index()
    peer_avg.rename(columns={'Excess_Return': 'Peer_Avg_Excess_Return'}, inplace=True)

    df_final = pd.merge(df_merged, peer_avg, on=peer_cols, how='left')

    def calc_sdi_core(group):
        fund_ret = group['Excess_Return']
        peer_ret = group['Peer_Avg_Excess_Return']
        count = len(fund_ret)
        if count < 26:
            return pd.Series({'Correlation': None, 'SDI': None, 'Obs_Count': count})

        corr = fund_ret.corr(peer_ret)
        return pd.Series({'Correlation': corr, 'SDI': 1 - corr, 'Obs_Count': count})

    sdi_results = df_final.groupby('MasterFundCode').apply(calc_sdi_core).reset_index()
    sdi_results = sdi_results.dropna(subset=['SDI'])

    # ==========================================
    # 5. 格式化并输出
    # ==========================================
    sdi_results['MasterFundCode'] = sdi_results['MasterFundCode'].astype(str).str.zfill(6)

    # 关联回原始描述信息
    final_output = pd.merge(sdi_results, df_info, on='MasterFundCode', how='left')

    # 按代码排序
    final_output = final_output.sort_values('MasterFundCode')

    final_output.to_excel(file_output, index=False)
    print(f"计算完成！\n结果已保存至: {file_output}")

if __name__ == "__main__":
    calculate_sdi_v7()

正在加载数据...
⚠️ 检测到【基金主要信息】中存在 247 条重复记录，已自动剔除。
正在处理日期并匹配利率...
正在计算同类基准收益及 SDI...
计算完成！
结果已保存至: C:\Users\qyw\Desktop\智能体\各智能体\基金经理分析智能体\数据\处理后\SDI\Fund_SDI_Results.xlsx
